# DeepWeeds — Nguyễn Thái Anh, 2A202602810

Notebook chạy trên Colab/Kaggle hoặc workspace local. Mặc định **kiểm tra bài nộp đã lưu bằng CPU**; bật `REPRODUCE=True` để thực hiện một thực nghiệm mới trên GPU theo thứ tự B → T → I → khóa final → test một lần → Excel/báo cáo.

Kết quả tham chiếu: final ConvNeXt-Tiny + CutMix + I07, seed 0/1/2. Xem `README.md`, `report.md` và `results.xlsx` của bài làm. Tái huấn luyện cần GPU, Internet, đủ thời gian và ổ lưu bền. Chọn runtime GPU trước khi bật `REPRODUCE`. Không đưa protocol/checkpoint của phiên cũ vào workspace mới.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPRODUCE = False
# Colab: có thể đổi thành /content/drive/MyDrive/deepweeds-lab và mount Drive trước.
WORKSPACE = Path('/content/deepweeds-lab') if Path('/content').exists() else Path.cwd()
REPOSITORY = 'https://github.com/nthanhwork/K4-Day2-NguyenThaiAnh-2A202602810.git'
if not (WORKSPACE / 'eval.py').exists():
    if WORKSPACE.exists() and any(WORKSPACE.iterdir()):
        raise RuntimeError('WORKSPACE phải là thư mục trống hoặc checkout của repo')
    subprocess.run(['git', 'clone', REPOSITORY, str(WORKSPACE)], check=True)
os.chdir(WORKSPACE)
SUB = WORKSPACE / 'submissions/2A202602810_NguyenThaiAnh'
CODE = SUB / 'code'
assert CODE.exists(), 'Cần push commit bài làm lên GitHub trước khi mở link Colab'
print('Workspace:', WORKSPACE, '| REPRODUCE:', REPRODUCE)

## Cài thư viện

Giữ torch/torchvision phù hợp CUDA runtime. Thực nghiệm gốc: Python 3.14.7, torch 2.10.0+cu128, torchvision 0.25.0, timm 1.0.30. Colab có thể có Python/runtime khác, nên đây là tái lập protocol, không cam kết kết quả giống từng bit. Trên local, dùng environment/lock file để khớp phiên bản.

In [ ]:
packages = ['numpy', 'pandas', 'Pillow', 'matplotlib', 'scikit-learn', 'openpyxl', 'nbformat']
if REPRODUCE:
    packages += ['timm==1.0.30', 'fvcore', 'nbclient', 'ipykernel']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
if REPRODUCE:
    import torch, torchvision
    assert torch.cuda.is_available(), 'Bật GPU runtime trước khi train'
    print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))

## Kiểm tra số liệu đã nộp

Chỉ đọc prediction/evidence, không load checkpoint hoặc forward test. Có thể dùng chế độ này để chấm lại trên CPU.

In [ ]:
if (SUB / 'results.xlsx').exists() and (SUB / 'evidence/source_index.json').exists():
    subprocess.run([sys.executable, str(CODE / 'check_submission.py')], check=True)
    import pandas as pd
    from IPython.display import display
    sheet = pd.read_excel(SUB / 'results.xlsx', sheet_name='Final')
    display(sheet[sheet.seed == 'mean'])
elif not REPRODUCE:
    raise FileNotFoundError('Bài nộp tham chiếu thiếu; mở checkout mới hoặc bật REPRODUCE để tiếp tục phiên mới')
else:
    import pandas as pd
    from IPython.display import display
    print('Phiên tái huấn luyện đang chạy; giữ protocol và tiếp tục các bước sau.')

## Chuẩn bị phiên tái huấn luyện và dữ liệu

Chỉ chạy nhánh dưới khi `REPRODUCE=True`. Giữ bản tham chiếu riêng, tạo prediction/curve mới; việc này tránh nhầm CSV test có sẵn với test của thực nghiệm mới. Thư mục `runs/` phải trống ở lần bắt đầu; các lần resume trong cùng workspace giữ nguyên protocol và marker. Các ảnh/CSV lấy từ tác giả, không chia lại hoặc sửa nhãn. ZIP được kiểm tra header và đường dẫn trước giải nén.

In [ ]:
import shutil
if REPRODUCE:
    reference = WORKSPACE / 'reference_submission'
    if not reference.exists():
        if (WORKSPACE / 'runs').exists() and any((WORKSPACE / 'runs').iterdir()):
            raise RuntimeError('Dùng checkout sạch cho thực nghiệm mới; không ghi đè runs có sẵn')
        reference.mkdir()
        for name in ('backbone_latency.json', 'backbone_latency.csv'):
            source = CODE / 'validation' / name
            if source.exists():
                shutil.move(str(source), str(reference / name))
        for name in ('predictions', 'curves', 'evidence', 'results.xlsx', 'report.md', 'report.pdf'):
            source = SUB / name
            if source.exists():
                shutil.move(str(source), str(reference / name))
        (SUB / 'predictions').mkdir()
        (SUB / 'curves').mkdir()
    labels = WORKSPACE / 'data/labels'
    labels.mkdir(parents=True, exist_ok=True)
    for source in (reference / 'evidence/labels').glob('*.csv'):
        target = labels / source.name
        if target.exists() and target.read_bytes() != source.read_bytes():
            raise ValueError('CSV hiện có khác nguồn tham chiếu')
        if not target.exists():
            shutil.copyfile(source, target)
    import urllib.request, zipfile
    images = WORKSPACE / 'data'
    if not all((images / name).exists() for name in pd.read_csv(labels / 'labels.csv').Filename):
        archive = images / 'images.zip'
        if not archive.exists():
            urllib.request.urlretrieve('https://zenodo.org/records/7939060/files/images.zip?download=1', archive)
        if not zipfile.is_zipfile(archive):
            raise RuntimeError('File tải về chưa phải images.zip hợp lệ; kiểm tra quota/response Zenodo')
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                target = (images / member.filename).resolve()
                if not target.is_relative_to(images.resolve()):
                    raise ValueError('ZIP chứa đường dẫn không an toàn')
            z.extractall(images)
        # Một số bản ZIP có thư mục images/ bên trong; dataset yêu cầu data/*.jpg.
        for image in list(images.rglob('*.jpg')):
            if image.parent != images:
                destination = images / image.name
                if destination.exists() and destination.read_bytes() != image.read_bytes():
                    raise ValueError('Ảnh trùng tên khác byte')
                if not destination.exists():
                    shutil.move(str(image), str(destination))
    subprocess.run([sys.executable, str(CODE / 'validate_setup.py'), '--decode-images'], check=True)

## Sanity và năm backbone

Ảnh overfit/sanity chỉ từ train. Chạy backbone tuần tự để dễ theo dõi trên GPU được cấp; không dùng thời gian train đồng thời làm latency benchmark. Checkpoint/metadata chứa tag thực tế; nếu timm đổi tag, dừng và kiểm tra thay vì giả định tương đương.

In [ ]:
if REPRODUCE:
    subprocess.run([sys.executable, str(CODE / 'sanity_checks.py')], check=True)
    sys.path.insert(0, str(CODE))
    import backbone_notebook as backbone
    for exp_id, (architecture, group) in backbone.BACKBONES.items():
        config = backbone.t00_config(exp_id, architecture)
        backbone.run_in_subprocess(config, group)

## Ablation và suy luận

T00 reuse B03. T01–T07 là ablation riêng; T08 khóa từ validation. I00/I01/I02/I07/I08 chỉ dùng val. Đo latency khi GPU không có compute process khác; dùng session mới của workspace này.

In [ ]:
if REPRODUCE:
    subprocess.run([sys.executable, str(CODE / 'ablation_notebook.py'), '--run', '--auto-t08'], check=True)
    subprocess.run([sys.executable, str(CODE / 'inference_notebook.py')], check=True)
    if not (CODE / 'validation/backbone_latency.json').exists():
        subprocess.run([sys.executable, str(CODE / 'submission_benchmark.py')], check=True)

## Final ba seed, test một lần và sản phẩm

Final helper được thiết kế cho T04+I07. Nếu kết quả sàng lọc của runtime mới chọn phương pháp khác, dừng và tạo protocol khác trước test, không sửa manifest đang dùng. Sau khi khóa, train đủ sáu mục (hai seed0 reuse), fit T trên val, seal checkpoint-set, test đúng một lần/model/seed. Không xóa marker để thử lại test. Chạy lại notebook đọc cache của test đã hoàn tất.

In [ ]:
if REPRODUCE:
    import json
    selected = json.loads((WORKSPACE / 'runs/inference/I_T04_seed0/selection.json').read_text())
    assert selected['quality_candidate'] == 'I07' and selected['realtime_candidate'] == 'I07', 'Runtime mới chọn phương pháp khác; cần protocol mới trước test'
    subprocess.run([sys.executable, str(CODE / 'final_notebook.py')], check=True)
    subprocess.run([sys.executable, str(CODE / 'export_submission.py'), '--snapshot'], check=True)
    subprocess.run([sys.executable, str(CODE / 'check_submission.py')], check=True)
    display(pd.read_excel(SUB / 'results.xlsx', sheet_name='Final'))

## Lưu kết quả

Lưu toàn bộ workspace, đặc biệt `runs/`, checkpoint, markers và raw cache vào ổ bền để resume. Bài nộp chỉ cần `submissions/2A202602810_NguyenThaiAnh/`, giữ evaluator và đề bài gốc trong repo. Không commit ảnh dữ liệu, môi trường, cache hoặc checkpoint lớn. Để mở notebook này trên Colab từ link GitHub, commit bài làm cần có trên nhánh `main` của repo công khai.